# Market Data Preprocessing for FuseChain

This notebook processes Ethereum historical market data from CoinMarketCap into daily aggregated features.

**Features extracted:**
- **Price**: Open, High, Low, Close (OHLC)
- **Volume & Market Cap**: Daily trading volume and market capitalization
- **Derived Features**:
  - Daily Return (% change)
  - Intraday Volatility ((High - Low) / Open)
  - 7-Day Rolling Volatility
- **Lag Features**: 1, 3, 7 day lags for key metrics

In [ ]:
import pandas as pd
import numpy as np
import os

## 1. Load Market Data

In [ ]:
input_path = '../data/raw/market/ethereum_historical_data_coinmarketcap.csv'

print("Loading market data...")
# Note: The CSV uses semicolon ';' as delimiter
df = pd.read_csv(input_path, sep=';')

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

## 2. Preprocessing & Cleaning

In [ ]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['day'] = df['timestamp'].dt.date

# Sort by date ascending
df = df.sort_values('day').reset_index(drop=True)

# Select and rename relevant columns
market_df = df[['day', 'open', 'high', 'low', 'close', 'volume', 'marketCap']].copy()
market_df.columns = ['day', 'eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'eth_market_cap']

# Verify date range
print(f"Date Range: {market_df['day'].min()} to {market_df['day'].max()}")

# Check for duplicates
print(f"Duplicates: {market_df['day'].duplicated().sum()}")
market_df.head()

## 3. Feature Engineering

In [ ]:
# 1. Daily Price Returns (Percentage Change)
market_df['eth_daily_return'] = market_df['eth_close'].pct_change().fillna(0)

# 2. Intraday Volatility (Normalized range)
market_df['eth_intraday_volatility'] = (market_df['eth_high'] - market_df['eth_low']) / market_df['eth_open']

# 3. 7-Day Rolling Volatility (Standard Deviation of Daily Returns)
market_df['eth_volatility_7d'] = market_df['eth_daily_return'].rolling(window=7).std().fillna(0)

# 4. Volume Change (Percentage Change)
market_df['eth_volume_change_pct'] = market_df['eth_volume'].pct_change().fillna(0)

print("Derived features created.")
market_df[['day', 'eth_daily_return', 'eth_volatility_7d', 'eth_intraday_volatility']].tail()

## 4. Generate Lag Features

In [ ]:
lag_features = ['eth_close', 'eth_volume', 'eth_daily_return', 'eth_volatility_7d']
lag_periods = [1, 3, 7]

for feature in lag_features:
    for lag in lag_periods:
        col_name = f"{feature}_lag{lag}"
        market_df[col_name] = market_df[feature].shift(lag)

# Fill initial NaNs caused by lagging (fill with 0 or backfill/forwardfill)
# Since these are continuous time series, forward fill then 0 is reasonable, 
# but simply filling 0 for missing history is safest for ML to avoid data leakage look-ahead implies
# actually, strict lag=NaN is best, but usually we fill 0 or mean.
market_df = market_df.fillna(0)

print(f"Final shape with lags: {market_df.shape}")
market_df.columns.tolist()

## 5. Save Processed Data

In [ ]:
output_dir = '../data/processed/market'
os.makedirs(output_dir, exist_ok=True)

# Save as Parquet
parquet_path = os.path.join(output_dir, 'market_daily_features.parquet')
market_df.to_parquet(parquet_path, index=False)
print(f"✅ Saved Parquet to: {parquet_path}")

# Save as CSV for inspection
csv_path = os.path.join(output_dir, 'market_daily_features.csv')
market_df.to_csv(csv_path, index=False)
print(f"✅ Saved CSV to: {csv_path}")

## 6. Verification

In [ ]:
print("Summary Statistics:")
market_df[['eth_daily_return', 'eth_volatility_7d', 'eth_volume']].describe()